# **Laborator 9** - *VoIP si Conferinta Audio/Video*

**Aplicatii Multimedia**  An Universitar 2025-2026

---

### **Cuprins**
1. [Transmisie UDP](#1)
2. [Calitatea retelei](#2)
3. [Bandwidth si codecuri](#3)
4. [Procesare audio](#4)
5. [Demo - Conferinta WebRTC](#5)


## **Instalare dependente**

Ruleaza celula de mai jos o singura data.


In [1]:
!pip install numpy scipy sounddevice matplotlib --quiet

---
## **1. Transmisie UDP <a id='1'></a>**

### *De ce UDP si nu TCP pentru VoIP?*

`TCP` garanteaza livrarea pachetelor, dar o face prin **retransmisie**. Intr-un apel vocal, un pachet intarziat cu mai mult de ~150ms este inutil - mai bine il ignoram decat sa asteptam retransmisia lui.

`UDP` nu garanteaza livrarea, dar are **latenta constanta si redusa**, ceea ce il face potrivit pentru audio si video in timp real.

### *Structura unui pachet RTP*

`RTP` (*Real-time Transport Protocol*) este protocolul standard pentru transmisia audio/video peste `UDP`. Headerul sau are urmatoarea structura:

```
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|V=2|P|X|  CC   |M|     PT      |       sequence number         |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                           timestamp                           |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|           synchronization source (SSRC) identifier            |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    payload (audio PCM)  ...                   |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
```

Campurile de interes pentru `VoIP`:
- **sequence number** (16 biti) - numarul pachetului, creste cu 1 la fiecare pachet
- **timestamp** (32 biti) - momentul de timp in samples fata de inceputul sesiunii
- **SSRC** (32 biti) - identifica sursa (cine trimite)
- **payload** - datele audio propriu-zise (PCM raw)


### **Exemplu** - *Construirea unui pachet*

Folosim `struct.pack` pentru a serializa headerul. Formatul `!HHI4s` inseamna:
- `!` - big-endian (network byte order)
- `H` - unsigned short, 2 bytes (flags)
- `H` - unsigned short, 2 bytes (sequence number)
- `I` - unsigned int, 4 bytes (timestamp)
- `4s` - 4 bytes (SSRC)


In [ ]:
import struct

HEADER_FORMAT = "!HHI4s"
HEADER_SIZE = struct.calcsize(HEADER_FORMAT)  # 12 bytes


def build_rtp_packet(seq, timestamp, payload, ssrc=b"LAB9"):
    # Primii 16 biti contin campurile RTP scurte:
    # V=2, P=0, X=0, CC=0, M=0, PT=0.
    # 0x8000 inseamna ca doar versiunea RTP este setata la 2.
    flags = 0x8000
    header = struct.pack(HEADER_FORMAT, flags, seq & 0xFFFF, timestamp, ssrc)
    return header + payload


# Construim un pachet de test
payload = b"\x00" * 320  # 160 samples PCM16 = 20ms la 8000 Hz
packet = build_rtp_packet(seq=0, timestamp=0, payload=payload)

print(f"Header size : {HEADER_SIZE} bytes")
print(f"Payload size: {len(payload)} bytes")
print(f"Total packet: {len(packet)} bytes")
print(f"Header hex  : {packet[:HEADER_SIZE].hex()}")

### **TODO 1** - *Parseaza un pachet primit*

Acum executa operatia inversa: primesti un sir de bytes si extrage campurile din header.

Completeaza functia `parse_rtp_packet` de mai jos.


In [ ]:
def parse_rtp_packet(data):
    # TODO 1a: dezambaleaza primii HEADER_SIZE bytes din data
    #   foloseste struct.unpack(HEADER_FORMAT, data[:HEADER_SIZE])
    #   returneaza un tuplu: (flags, seq, timestamp, ssrc)

    # TODO 1b: extrage payload-ul
    #   payload = tot ce e dupa primii HEADER_SIZE bytes

    # TODO 1c: returneaza un dict cu cheile: 'seq', 'timestamp', 'ssrc', 'payload'
    pass


# Test - ar trebui sa printeze seq=0, timestamp=0, ssrc=b'LAB9'
result = parse_rtp_packet(packet)
print(result)

### **Exemplu** - *Sender UDP*

Trimitem 10 pachete pe localhost, portul `5005`.
Fiecare pachet contine: un payload de zeros (`\x00`), un header cu sequence number si timestamp incrementale.

#### *Ce este un socket UDP*?

Un socket UDP este un obiect software pus la dispozitie de sistemul de operare pentru a abstractiza complet complexitatea transmisiunilor de retea, permitand aplicatiei tale VoIP sa expedieze si sa receptioneze pachete audio cu latenta minima.

Functioneaza ca un `endpoint` definit printr-o **adresa IP** si un **port specific**. Socket-ul preia toata munca grea (conversie, impachetare si rutare a bitilor la nivel hardware) si ofera o interfata curata prin care datele pot fi transferate.

#### *Parametrii Constructor:*
| Constanta | Descriere |
| :--- | :--- |
| `AF_INET` | Indica familia de adrese **IPv4**. |
| `SOCK_DGRAM` | Defineste un socket de tip **Datagram** (specific protocolului UDP). |

#### *Metode:*
*   `bind((ip, port))`: Asociaza socket-ul cu o adresa si un port specific (utilizat in principal pe partea de server/receptor).
*   `sendto(data, (ip, port))`: Transmite un pachet catre destinatie, **fara a necesita o conexiune anterioara**.
*   `recvfrom(bufsize)`: Receptioneaza datele si returneaza un tuplu `(date, adresa_sursa)`.

In [ ]:
import socket
import time

SAMPLE_RATE = 8000  # Hz
CHUNK_SIZE = 160  # samples per packet, 20ms

def run_sender(host="127.0.0.1", port=5005, n_packets=10):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    seq = 0
    timestamp = 0

    for i in range(n_packets):
        # Payload de test: repetam numarul de secventa ca bytes,
        # ca sa putem recunoaste usor pachetul la receiver.
        payload = (seq).to_bytes(2, "big") * CHUNK_SIZE

        packet = build_rtp_packet(seq, timestamp, payload)
        
        # Trimitem pachetul catre adresa si portul receiver-ului.
        # sendto este specific UDP, unde nu avem conexiune persistenta.
        sock.sendto(packet, (host, port))

        print(f"Sent packet seq={seq}, timestamp={timestamp}")
        seq += 1
        timestamp += CHUNK_SIZE
        time.sleep(0.02)  # 20ms intre pachete

    sock.close()


# Vom incepe trimiterea de pachete dupa ce pornim receiver-ul in TODO 2

### **TODO 2** - *Receiver UDP*

Completeaza functia `run_receiver`.
Comportament:
1. Crează un socket UDP, legat la portul `5005`.
2. Primește `n_packets` pachete.
3. Parsează fiecare pachet cu funcția din `TODO1`.
4. Afișează `seq` și `timestamp` pentru fiecare pachet primit.


In [ ]:
def run_receiver(host="127.0.0.1", port=5005, n_packets=10):
    # TODO 2a: creeaza un socket UDP
    #   sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    # TODO 2b: leaga socketul la adresa si port ca sa asculte pachete
    #   sock.bind((host, port))

    # TODO 2c: primeste n_packets pachete intr-un loop
    #   - data, address = sock.recvfrom(4096)
    #   - parseaza fiecare pachet: pkt = parse_rtp_packet(data)
    #   - printeaza seq si timestamp: pkt["seq"], pkt["timestamp"]

    # TODO 2d: inchide socketul
    #   sock.close()
    pass


# Pornim receiver-ul intr-un thread separat
import threading

t = threading.Thread(target=run_receiver, args=("127.0.0.1", 5005, 10), daemon=True)
t.start()
time.sleep(5)  # dam timp receiver-ului sa porneasca
run_sender()
t.join(timeout=5)

---
## **2. Calitatea retelei <a id='2'></a>**

### *Packet loss*

Intr-o transmisie `VoIP`, pachetele pot fi pierdute pe drum. Le detectam comparand numerele de secventa - daca sequence number-ul nu este consecutiv, inseamna ca un pachet a fost pierdut.

Formula pentru procentul de pierderi:

$$\text{loss\%} = \frac{\text{pachete pierdute}}{\text{pachete trimise}} \times 100$$

Un apel VoIP este considerabil afectat la **loss > 5%**.


### **Exemplu** - *Detectare pachete pierdute*


In [ ]:
# Simulam o secventa in care lipsesc pachetele 3 si 7
example_sequence = [0, 1, 2, 4, 5, 6, 8, 9]

# Comparam fiecare pachet primit cu pachetul anterior
# ca sa verificam daca avem o secventa continua sau daca lipsesc pachete
def count_losses(received_seqs):
    lost = 0
    gaps = []
    for i in range(1, len(received_seqs)):
        expected = received_seqs[i - 1] + 1
        actual = received_seqs[i]
        if actual != expected:
            gap = actual - expected
            lost += gap
            gaps.append((expected, actual - 1))
    total = received_seqs[-1] - received_seqs[0] + 1
    loss_pct = lost / total * 100
    return lost, loss_pct, gaps


lost, pct, gaps = count_losses(example_sequence)
print(f"Pachete pierdute: {lost}")
print(f"Packet loss     : {pct:.1f}%")
print(f"Gap-uri detectate: {gaps}")  # (primul_lipsa, ultimul_lipsa)

### **TODO 3** - *Calculeaza packet loss*

Folosind functia `count_losses` de mai sus, calculeaza packet loss-ul pentru secventa de mai jos si determina daca apelul a fost afectat (`loss > threshold`).


In [ ]:
received = [0, 1, 2, 3, 5, 6, 7, 8, 9, 13, 14, 15, 16, 17, 18, 19]

# TODO 3a: apeleaza count_losses(received)
#   returneaza (lost, pct, gaps)
#   - lost: numarul de pachete lipsa
#   - pct: procentul de pierderi (0-100)
#   - gaps: lista de (primul_lipsa, ultimul_lipsa)

# TODO 3b: printeaza numarul de pachete pierdute si procentul de pierderi

# TODO 3c: printeaza intervalele lipsa (lista gaps)

# TODO 3d: apelul este afectat?
#   pierdere > 5% este considerata semnificativa
#   printeaza "DA" sau "NU" cu threshold-ul

### **Jitter**

**Jitter** = variatia in timp a intarzierii pachetelor.

Daca pachetele ajung la intervale neregulate, audio-ul suna discontinuu. Il masuram ca **deviatia standard** a intervalelor dintre pachete (**RMSE** - *Root Mean Squared Error*):

$$\text{jitter} = \sigma(\Delta t) = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\Delta t_i - \overline{\Delta t})^2}$$

Unde $\Delta t_i$ este intervalul de timp dintre pachetul $i$ si $i-1$.

Un jitter sub **30ms** este considerat acceptabil pentru VoIP.


### **Exemplu** - *Calcul jitter*


In [ ]:
import numpy as np

# Timestamps de sosire (in milisecunde) - ideal ar fi la fiecare 20ms
example_timestamps = [0, 20, 41, 19, 40, 62, 79, 101, 118, 140]


def calculate_jitter(arrival_times_ms):
    intervals = np.diff(arrival_times_ms)  # diferentele consecutive
    jitter = np.std(intervals)  # deviatia standard
    return jitter, intervals


jitter, intervals = calculate_jitter(example_timestamps)
print(f"Intervale (ms): {intervals}")
print(f"Interval mediu: {np.mean(intervals):.1f} ms  (ideal: 20ms)")
print(f"Jitter        : {jitter:.2f} ms")

### **TODO 4** - *Calculeaza jitter pentru un apel real*

Mai jos ai timestamps-urile de sosire ale pachetelor dintr-un apel cu conditii proaste de retea. Calculeaza jitter-ul si spune daca apelul este afectat.


In [ ]:
arrival_times = [
    0,
    22,
    38,
    65,
    81,
    104,
    118,
    155,
    172,
    190,
    215,
    228,
    261,
    279,
    301,
    318,
    356,
    374,
    390,
    421,
]

# TODO 4a: apeleaza calculate_jitter(arrival_times)
#   returneaza (jitter, intervals)
#   - jitter: deviatia standard a intervalelor dintre pachete (ms)
#   - intervals: array cu diferentele dintre timestamp-uri consecutive

# TODO 4b: printeaza valoarea jitter-ului
#   este acceptabil? jitter < 30ms = acceptabil pentru VoIP

# TODO 4c: traseaza intervalele cu matplotlib
#   - plt.figure(figsize=(10, 3))
#   - plt.plot(intervals, marker='o', color="coral", linewidth=1.5) pentru fiecare interval
#   - plt.axhline(20, color='green', linestyle='--', label='Ideal (20ms)')
#   - plt.axhline(30, color='red',   linestyle='--', label='Limita (30ms)')
#   - plt.fill_between(range(len(intervals)), intervals, 20, alpha=0.15, color="coral")
#   - adauga titlu, etichete axe, legenda, grid si plt.show()
import matplotlib.pyplot as plt

---
## **3. Bitrate, bandwidth si codecuri <a id='3'></a>**

### *Bitrate vs bandwidth*

Un apel VoIP genereaza un flux continuu de pachete audio.

**Bitrate-ul** reprezinta cati biti pe secunda produce semnalul audio sau codecul.

**Bandwidth-ul** reprezinta capacitatea retelei sau traficul total consumat pe retea. In practica, bandwidth-ul necesar este mai mare decat bitrate-ul audio, deoarece fiecare pachet mai are headere: `RTP`, `UDP`, `IP` si `Ethernet`.

In acest laborator calculam mai intai bitrate-ul audio brut, apoi estimam cate apeluri incap intr-o conexiune de 5 Mbps. Pentru simplitate, ignoram overhead-ul de retea.

### *Formula bitrate audio brut*

$$\text{bitrate (bps)} = \text{sample\_rate} \times \text{bits\_per\_sample} \times \text{channels}$$

Codecuri / formate audio comune:

| Codec / format | Sample rate | Bits/sample | Channels | Bitrate brut |
|---|---:|---:|---:|---:|
| PCM 16-bit telefonie | 8000 Hz | 16 | 1 | 128 kbps |
| G.711 u-law | 8000 Hz | 8 | 1 | 64 kbps |
| PCM 16-bit wideband | 16000 Hz | 16 | 1 | 256 kbps |
| PCM stereo CD | 44100 Hz | 16 | 2 | 1411 kbps |

### **Exemplu** - *Calcul pentru PCM 16-bit la 8000 Hz*


In [ ]:
def calc_bitrate(sample_rate, bits_per_sample, channels):
    return sample_rate * bits_per_sample * channels


def calc_mb_per_hour(bitrate_bps):
    bytes_per_hour = bitrate_bps / 8 * 3600
    return bytes_per_hour / (1024**2)


sr, bits, ch = 8000, 16, 1
bitrate = calc_bitrate(sr, bits, ch)

print(f"PCM 16-bit 8kHz mono")
print(f"  Bitrate      : {bitrate} bps = {bitrate / 1000:.0f} kbps")
print(f"  Date / ora   : {calc_mb_per_hour(bitrate):.1f} MB")

### **TODO 5** - *Completeaza tabelul pentru codecuri*

Pentru fiecare codec / format audio, calculeaza:

1. bitrate-ul brut;
2. volumul de date generat intr-o ora;
3. cate apeluri simultane incap intr-o conexiune de **5 Mbps**.

Pentru simplitate, ignoram overhead-ul `RTP`/`UDP`/`IP`/`Ethernet`.

#### *Formule*:

$$\text{bitrate (bps)} = \text{sample\_rate} \times \text{bits\_per\_sample} \times \text{channels}$$

$$\text{apeluri simultane} = \left\lfloor \frac{\text{bandwidth total (bps)}}{\text{bitrate codec (bps)}} \right\rfloor$$

In [ ]:
codecuri = {
    "PCM 16-bit 8kHz mono": (8000, 16, 1),
    "G.711 ulaw 8kHz mono": (8000, 8, 1),
    "PCM 16-bit 16kHz mono": (16000, 16, 1),
    "PCM 16-bit 44kHz stereo": (44100, 16, 2),
}

bandwidth_total_bps = 5_000_000  # 5 Mbps

# TODO 5:
# Pentru fiecare codec din dictionar:
#   TODO 5a: calculeaza bitrate = calc_bitrate(sr, bits, ch)
#   TODO 5b: calculeaza mb = calc_mb_per_hour(bitrate)
#   TODO 5c: calculeaza apeluri = bandwidth_total_bps // bitrate
#   TODO 5d: printeaza un tabel cu:
#       - numele codec-ului
#       - bitrate-ul in kbps
#       - datele generate intr-o ora
#       - numarul de apeluri simultane intr-o conexiune de 5 Mbps
#
# Hint:
#   for nume, (sr, bits, ch) in codecuri.items():
#       ...
#
# Exemplu de rand:
#   print(f"{nume:<28} {bitrate/1000:>8.0f} kbps {mb:>8.1f} MB {apeluri:>8}")

---
## **4. Procesare audio <a id='4'></a>**

### *Banda vocala telefonica*

Vocea umana inteligibila se afla aproximativ in gama **300 Hz - 3400 Hz**. Aceasta este banda folosita traditional in telefonie, inclusiv in PSTN si G.711. Frecventele din afara acestui interval pot exista in voce, dar nu sunt esentiale pentru intelegerea vorbirii.

### *Filtru Butterworth*

Un **filtru Butterworth** este un tip de filtru proiectat sa aiba un raspuns cat mai plat (fara fluctuatii) in banda de trecere. Asta inseamna ca nu amplifica sau taie artificial anumite frecvente din zona pe care vrem sa o pastram.

Patratul amplitudinii unui filtru Butterworth de ordinul $n$ este descris de formula:

$$ |H(j\omega)|^2 = \frac{1}{1 + \left(\frac{\omega}{\omega_c}\right)^{2n}} $$

Unde:
*   $|H(j\omega)|$ este amplitudinea filtrului.
*   $n$ este **ordinul filtrului**.
*   $\omega$ este frecventa semnalului la un moment dat.
*   $\omega_c$ este frecventa de taiere.

Folosim un filtru Butterworth deoarece este simplu de implementat, nu distorsioneaza faza semnalului la fel de mult ca alte filtre (vezi `Chebyshev`) si este foarte stabil pentru prelucrarea audio.

```python
sos = scipy.signal.butter(
    N=4,
    Wn=[f_low, f_high],
    btype="bandpass",
    fs=sample_rate,
    output="sos"
)

filtered = scipy.signal.sosfilt(sos, audio)

### **Exemplu** - *Aplicare filtru lowpass la 4000 Hz*

Mai intai capturam 3 secunde de audio, apoi aplicam un **filtru trece-jos** (`FTJ`) la 4000 Hz si comparam spectrogramele.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sounddevice as sd
from scipy import signal as sp

SR = 44100

print("Captureaza 3 secunde... vorbeste!")
audio = sd.rec(int(3 * SR), samplerate=SR, channels=1, dtype="float32")
sd.wait()
audio = audio.flatten()
print("Gata.")

# Filtru lowpass la 4000 Hz
sos_low = sp.butter(N=4, Wn=4000, btype="lowpass", fs=SR, output="sos")
filtered_low = sp.sosfilt(sos_low, audio)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].specgram(audio, Fs=SR, NFFT=1024, cmap="plasma")
axes[0].set_title("Original")
axes[0].set_ylabel("Frecventa (Hz)")
axes[0].set_xlabel("Timp (s)")
axes[1].specgram(filtered_low, Fs=SR, NFFT=1024, cmap="plasma")
axes[1].set_title("Dupa filtru lowpass 4000 Hz")
axes[1].set_xlabel("Timp (s)")
plt.tight_layout()
plt.show()

### **TODO 6** - *Aplica filtrul bandpass telefonic*

Acum aplica un filtru **bandpass 300-3400 Hz** pe acelasi semnal audio si compara spectrograma cu originalul.

Observa cum dispar frecventele sub 300 Hz si cele peste 3400 Hz.


In [ ]:
# TODO 6a: proiecteaza filtrul Butterworth bandpass pentru voce telefonica (300-3400 Hz)
#   sos_band = sp.butter(N=4, Wn=[300, 3400], btype='bandpass', fs=SR, output='sos')
#   N=4 ordinul filtrului (taiere mai brusca decat N=2)
#   Wn=[300, 3400] = frecventele de taiere in Hz
sos_band = None  # inlocuieste cu apelul butter() de mai sus

# TODO 6b: aplica filtrul pe semnalul audio capturat
#   filtered_band = sp.sosfilt(sos_band, audio)
filtered_band = None  # inlocuieste cu apelul sosfilt() de mai sus

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].specgram(audio, Fs=SR, NFFT=1024, cmap="plasma")
axes[0].set_title("Original")
axes[0].set_ylabel("Frecventa (Hz)")
axes[0].set_xlabel("Timp (s)")
axes[1].specgram(filtered_band, Fs=SR, NFFT=1024, cmap="plasma")
axes[1].set_title("Dupa filtru bandpass 300-3400 Hz")
axes[1].set_xlabel("Timp (s)")
plt.tight_layout()
plt.show()

---
## **5. Demo - Conferinta WebRTC <a id='5'></a>**

Tot ce am prezentat pana acum - *transmisie UDP*, *packet loss*, *jitter*, *bandwidth*, *procesare audio* - sta la baza unei aplicatii de conferinta reala.

Serverele din acest laborator implementeaza exact aceste concepte:
- `signaling_server.py` - coordoneaza conexiunile WebRTC (via WebSocket pur, fara socket.io)
- `ws_processing_server.py` - aplica filtre audio/video in Python (FastAPI)
- `web/` - interfata browser cu WebRTC nativ, fara dependente externe

### *Cum pornesti*

Deschide un terminal (sau Anaconda Prompt pe Windows). Din folderul **`main`** (acolo este `requirements.txt`), ruleaza:

```bash
pip install -r requirements.txt   # o singura data
python src/client-server/start.py
```

Apoi deschide **http://localhost:4321** in doua tab-uri sau de pe doua calculatoare din aceeasi retea.

Vei vedea:
- Stream video live de la camera fiecarui participant.
- Filtre video aplicate in Python (**blur**, **grayscale**, **edges**, **cartoon**, **sepia**).
- Filtrul bandpass telefonic pe audio.
- Metrici QoS in timp real: **RTT**, **jitter**, **packet loss**.